Load train and test datasets

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.preprocessing import preprocess_complaint

In [2]:
train_df = pd.read_parquet('../data/processed/train_complaints.parquet')
test_df = pd.read_parquet('../data/processed/test_complaints.parquet')

In [3]:
print(train_df.shape)
print(test_df.shape)

(812720, 4)
(203180, 4)


In [4]:
train_df.head()

,Complaint ID,Consumer complaint narrative,Department,Priority
0,7251882,This is regarding the Texas B-On-Time Student ...,Student loan,high_priority
1,16800276,"On top of other discrepancies, XXXX XXXX accou...",Credit reporting,standard
2,2292018,There was fraudulent activity on my account wh...,Bank accounts,high_priority
3,7835057,I have consistently ensured my payments are ma...,Card services,standard
4,4313529,I tried to send them a letter last XX/XX/2020....,Mortgage,standard


Define features and targets

In [5]:
X_train = train_df["Consumer complaint narrative"]
X_test  = test_df["Consumer complaint narrative"]

y_train_dept = train_df["Department"]
y_test_dept  = test_df["Department"]

y_train_priority = train_df["Priority"]
y_test_priority  = test_df["Priority"]

Preprocess text

In [6]:
X_train_clean = [preprocess_complaint(x) for x in X_train]
X_test_clean  = [preprocess_complaint(x) for x in X_test]

Encode Labels

In [7]:
from sklearn.preprocessing import LabelEncoder

le_dept = LabelEncoder()
y_train_dept_enc = le_dept.fit_transform(y_train_dept)
y_test_dept_enc  = le_dept.transform(y_test_dept)

le_priority = LabelEncoder()
y_train_priority_enc = le_priority.fit_transform(y_train_priority)
y_test_priority_enc  = le_priority.transform(y_test_priority)

Tokenization

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256
    )

train_encodings = tokenize(X_train_clean)
test_encodings  = tokenize(X_test_clean)

Create Dataset

In [9]:
import torch

class ComplaintDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

Build datasets - Department

In [10]:
train_dataset = ComplaintDataset(train_encodings, y_train_dept_enc)
test_dataset  = ComplaintDataset(test_encodings, y_test_dept_enc)

 Load Model

In [11]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(le_dept.classes_)
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training setup

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",  # <-- important fix
    save_strategy="epoch",
    load_best_model_at_end=True
)

Metrics

In [13]:
from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

Train

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/Users/mikael_abyssinia/complaint-routing-ml/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

Evaluate

In [ ]:
trainer.evaluate()

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

model_name = "distilbert-base-uncased"

# ---- APPLY PREPROCESS FUNCTION ----
X_train_clean = [preprocess_complaint(t) for t in X_train]
X_test_clean  = [preprocess_complaint(t) for t in X_test]

# ---- TOKENIZER ----
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="./model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",           
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1"
)

# ---- DEPARTMENT ----
train_ds_dept = Dataset.from_dict({"text": X_train_clean, "label": y_train_dept})
test_ds_dept  = Dataset.from_dict({"text": X_test_clean,  "label": y_test_dept})

train_ds_dept = train_ds_dept.map(tokenize, batched=True)
test_ds_dept  = test_ds_dept.map(tokenize, batched=True)

train_ds_dept.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds_dept.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

model_dept = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(set(y_train_dept))
)

trainer_dept = Trainer(
    model=model_dept,
    args=args,
    train_dataset=train_ds_dept,
    eval_dataset=test_ds_dept,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer_dept.train()

y_pred_dept = np.argmax(trainer_dept.predict(test_ds_dept).predictions, axis=1)
print("DEPARTMENT REPORT\n", classification_report(y_test_dept, y_pred_dept))
print("DEPARTMENT MACRO F1:", f1_score(y_test_dept, y_pred_dept, average="macro"))

# ---- PRIORITY ----
train_ds_pri = Dataset.from_dict({"text": X_train_clean, "label": y_train_priority})
test_ds_pri  = Dataset.from_dict({"text": X_test_clean,  "label": y_test_priority})

train_ds_pri = train_ds_pri.map(tokenize, batched=True)
test_ds_pri  = test_ds_pri.map(tokenize, batched=True)

train_ds_pri.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds_pri.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

model_pri = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(set(y_train_priority))
)

trainer_pri = Trainer(
    model=model_pri,
    args=args,
    train_dataset=train_ds_pri,
    eval_dataset=test_ds_pri,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer_pri.train()

y_pred_pri = np.argmax(trainer_pri.predict(test_ds_pri).predictions, axis=1)
print("\nPRIORITY REPORT\n", classification_report(y_test_priority, y_pred_pri))
print("PRIORITY MACRO F1:", f1_score(y_test_priority, y_pred_pri, average="macro"))